In [23]:
import tkinter as tk
from tkinter import ttk
import socket
import queue
import threading
import logging
import sys
import cv2
import numpy as np
from datetime import datetime
from collections import deque
from pathlib import Path
import time
from PIL import Image as PILImage
from PIL import ImageTk
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import pandas as pd
import seaborn as sns
import mediapipe as mp
import json
import struct

In [24]:
def send_number_to_nao(number):

    SERVER_IP = "127.0.0.1"
    SERVER_PORT = 9998

    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as client_socket:
        client_socket.connect((SERVER_IP, SERVER_PORT))
        client_socket.send(str(number).encode('utf-8'))

In [25]:
class PoseAnalytics:
    """Analytics component for tracking and analyzing pose detection metrics"""
    def __init__(self, save_dir='analytics'):
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(exist_ok=True)
        
        # Initialize analytics storage
        self.session_data = {
            'timestamps': [],
            'postures': [],
            'confidence_scores': [],
            'processing_times': [],
            'fps_values': []
        }
        
        # Setup logging
        logging.basicConfig(
            filename=self.save_dir / 'pose_detection.log',
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s'
        )
        
        # Initialize real-time metrics
        self.current_metrics = {
            'session_start': datetime.now(),
            'poses_detected': 0,
            'average_fps': 0.0
        }
    
    def update_metrics(self, posture, confidence, process_time, fps):
        
        timestamp = datetime.now()
        
        # Update session data
        self.session_data['timestamps'].append(timestamp)
        self.session_data['postures'].append(posture)
        self.session_data['confidence_scores'].append(confidence)
        self.session_data['processing_times'].append(process_time)
        self.session_data['fps_values'].append(fps)
        
        # Update real-time metrics
        self.current_metrics['poses_detected'] += 1
        self.current_metrics['average_fps'] = np.mean(self.session_data['fps_values'])
        
        # Log significant events
        if confidence < 0.5:
            logging.warning(f"Low confidence detection: {confidence:.2f}")
    
    def _create_posture_distribution_plot(self, df):
        
        plt.figure(figsize=(10, 6))
        posture_counts = df['postures'].value_counts()
        sns.barplot(x=posture_counts.values, y=posture_counts.index)
        plt.title('Posture Distribution')
        plt.xlabel('Count')
        plt.tight_layout()
        plt.savefig(self.save_dir / 'posture_distribution.png')
        plt.close()

    def _create_performance_plot(self, df):
        
        plt.figure(figsize=(12, 6))
        plt.plot(df['timestamps'], df['fps_values'], label='FPS')
        plt.plot(df['timestamps'], df['processing_times'], label='Processing Time')
        plt.title('Performance Metrics Over Time')
        plt.xlabel('Time')
        plt.ylabel('Value')
        plt.legend()
        plt.tight_layout()
        plt.savefig(self.save_dir / 'performance_metrics.png')
        plt.close()

    def _create_confidence_plot(self, df):
        
        plt.figure(figsize=(10, 6))
        plt.plot(df['timestamps'], df['confidence_scores'])
        plt.title('Detection Confidence Over Time')
        plt.xlabel('Time')
        plt.ylabel('Confidence Score')
        plt.tight_layout()
        plt.savefig(self.save_dir / 'confidence_scores.png')
        plt.close()

    def generate_analytics_report(self):
        
        if not self.session_data['timestamps']:
            logging.warning("No data collected for analytics report")
            return

        # Convert data to DataFrame
        df = pd.DataFrame(self.session_data)
        
        # Generate visualizations
        self._create_posture_distribution_plot(df)
        self._create_performance_plot(df)
        self._create_confidence_plot(df)
        
        # Generate summary statistics
        summary = self._generate_summary_stats(df)
        
        # Save report
        self._save_analytics_report(df, summary)
    
    def _generate_summary_stats(self, df):
        
        summary = {
            'total_frames': len(df),
            'average_fps': df['fps_values'].mean(),
            'average_confidence': df['confidence_scores'].mean(),
            'most_common_posture': df['postures'].mode().iloc[0] if not df['postures'].empty else 'None',
            'session_duration': str(datetime.now() - self.current_metrics['session_start'])
        }
        return summary

    def _save_analytics_report(self, df, summary):
        """Save analytics report to file"""
        report_path = self.save_dir / f'analytics_report_{datetime.now().strftime("%Y%m%d_%H%M%S")}.txt'
        with open(report_path, 'w') as f:
            f.write("=== Pose Detection Analytics Report ===\n\n")
            for key, value in summary.items():
                f.write(f"{key}: {value}\n")

In [26]:
class PoseDetector:
    def __init__(self):
        # MediaPipe initialization
        self.mp_pose = mp.solutions.pose
        self.pose = self.mp_pose.Pose(
            min_detection_confidence=0.7,
            min_tracking_confidence=0.7,
            model_complexity=2
        )
        self.mp_drawing = mp.solutions.drawing_utils
        self.mp_drawing_styles = mp.solutions.drawing_styles
        
        # Initialize analytics
        self.analytics = PoseAnalytics()
        
        # Performance monitoring
        self.fps_history = deque(maxlen=30)
        self.detection_history = deque(maxlen=30)
        self.joint_consistency = deque(maxlen=30)
        
        # Movement tracking
        self.previous_landmarks = None
        self.movement_history = deque(maxlen=10)
        
        # Initialize current state attributes
        self.current_confidence = 0.0
        self.current_landmarks = None
        self.current_frame = None
        
        # Confidence tracking
        self.confidence_history = deque(maxlen=10)
        self.low_confidence_count = 0
        self.max_low_confidence_attempts = 3
        
        # Threshold values
        self.movement_threshold = 0.02  # Normalized movement threshold
        self.head_movement_threshold = 0.015
        self.shoulder_movement_threshold = 0.018
        self.squat_threshold = 0.25
        self.standing_hip_knee_ratio = 0.9
        self.sitting_hip_knee_ratio = 0.4
        
        # Confidence thresholds
        self.low_confidence_threshold = 0.5
        self.very_low_confidence_threshold = 0.3
        
        # Movement smoothing
        self.shoulder_positions = deque(maxlen=5)
        self.head_positions = deque(maxlen=5)
        
        # Posture change tracking
        self.last_posture_change_time = 0
        self.posture_change_interval = 0.5  # 1 seconds between posture changes
        self.current_posture = None
        
        
        # Initialize other attributes
        self.current_frame_shape = None
        self.recording = False
        self.video_writer = None
        self.output_dir = Path('output')
        self.output_dir.mkdir(exist_ok=True)
        
    def process_frame(self, frame):

        try:
            start_time = time.time()
            if frame is None or frame.size == 0:
                logging.error("Invalid frame received")
                return frame

            self.current_frame_shape = frame.shape
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # Process with MediaPipe
            results = self.pose.process(rgb_frame)
            landmarks = None
            confidence = 0.0

            if results.pose_landmarks:
                landmarks = results.pose_landmarks
                confidence = np.mean([lm.visibility for lm in landmarks.landmark])
                
                # Update confidence history
                self.confidence_history.append(confidence)
                
                # Check confidence levels
                confidence_status = self._check_confidence(confidence)
                
                if confidence_status == "low":
                    
                    
                    # Add low confidence warning to frame
                    cv2.putText(frame, "LOW CONFIDENCE: Move back", 
                                (10, self.current_frame_shape[0] - 50), 
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
                    
                if confidence_status == "very_low":
                    
                    
                    # Add very low confidence warning to frame
                    cv2.putText(frame, "VERY LOW CONFIDENCE: Please reposition!", 
                                (10, self.current_frame_shape[0] - 50), 
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
                    
                    # Optional: Add blinking effect
                    if int(time.time() * 2) % 2 == 0:
                        frame = cv2.addWeighted(frame, 0.5, np.zeros_like(frame), 0.5, 0)

                try:
                    # Detect posture
                    posture = self.detect_posture(landmarks)
                    self.current_posture = posture
                    self.current_confidence = confidence
                    self.current_landmarks = landmarks

                    # Calculate and draw bounding box
                    bbox = self.calculate_bounding_box(landmarks, model_type="mediapipe")
                    self.draw_action_box(frame, bbox, posture)

                    # Draw landmarks
                    self.mp_drawing.draw_landmarks(
                        frame,
                        landmarks,
                        self.mp_pose.POSE_CONNECTIONS,
                        landmark_drawing_spec=self.mp_drawing_styles.get_default_pose_landmarks_style()
                    )

                    # Calculate metrics
                    process_time = time.time() - start_time
                    fps = 1 / process_time if process_time > 0 else 0
                    self.fps_history.append(fps)

                    # Update analytics
                    self.analytics.update_metrics(
                        posture=posture,
                        confidence=confidence,
                        process_time=process_time,
                        fps=fps
                    )
                except Exception as e:
                    logging.error(f"Error processing landmarks: {str(e)}")

            
            # Draw performance metrics
            self._draw_performance_metrics(frame, fps if 'fps' in locals() else 0, confidence)

            # Handle recording if active
            if self.recording and self.video_writer is not None:
                try:
                    self.video_writer.write(frame)
                except Exception as e:
                    logging.error(f"Error writing video frame: {str(e)}")

            return frame

        except Exception as e:
            logging.error(f"Error in process_frame: {str(e)}")
            return frame

    def _draw_performance_metrics(self, frame, fps, confidence):
        
        # Calculate average FPS
        avg_fps = np.mean(self.fps_history) if self.fps_history else 0
        
        # Draw metrics
        metrics_text = [
            f"FPS: {avg_fps:.1f}",
            f"Confidence: {confidence:.2f}",
            f"Posture: {self.current_posture}"
        ]
        
        y_position = 30
        for text in metrics_text:
            cv2.putText(frame, text, (10, y_position),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            y_position += 25

    def toggle_recording(self):
 
        if not self.recording:
            # Start recording
            filename = self.output_dir / f'pose_recording_{datetime.now().strftime("%Y%m%d_%H%M%S")}.avi'
            fourcc = cv2.VideoWriter_fourcc(*'XVID')
            self.video_writer = cv2.VideoWriter(str(filename), fourcc, 20.0, 
                                              (self.current_frame_shape[1], self.current_frame_shape[0]))
            self.recording = True
            logging.info("Recording started")
        else:
            # Stop recording
            if self.video_writer is not None:
                self.video_writer.release()
                self.video_writer = None
            self.recording = False
            logging.info("Recording stopped")

    def save_frame(self):

        if self.current_frame_shape is not None:
            filename = self.output_dir / f'pose_frame_{datetime.now().strftime("%Y%m%d_%H%M%S")}.jpg'
            cv2.imwrite(str(filename), self.current_frame)
            logging.info(f"Frame saved: {filename}")

    def _load_gesture_patterns(self):

        return {
            'wave': {'sequence': ['standing with raised arms', 'standing'], 'threshold': 0.8},
            'squat': {'sequence': ['standing', 'squatting', 'standing'], 'threshold': 0.9}
        }

    def _detect_head_movement(self, landmarks):

        nose = landmarks.landmark[self.mp_pose.PoseLandmark.NOSE]
        left_ear = landmarks.landmark[self.mp_pose.PoseLandmark.LEFT_EAR]
        right_ear = landmarks.landmark[self.mp_pose.PoseLandmark.RIGHT_EAR]
        
        # Store current head position
        head_pos = (nose.x, nose.y)
        self.head_positions.append(head_pos)
        
        if len(self.head_positions) < 2:
            return ""
            
        # Calculate movement
        prev_pos = self.head_positions[-2]
        dx = head_pos[0] - prev_pos[0]
        dy = head_pos[1] - prev_pos[1]
        
        # Calculate ear position relative to nose for head rotation
        left_ear_dist = abs(left_ear.x - nose.x)
        right_ear_dist = abs(right_ear.x - nose.x)
        ear_ratio = left_ear_dist / right_ear_dist if right_ear_dist > 0 else 1.0
        
        # Detect movements
        if abs(dx) > self.head_movement_threshold:
            if ear_ratio > 1.2:
                return "turning head left"
            elif ear_ratio < 0.8:
                return "turning head right"
            else:
                return "moving head horizontally"
        elif abs(dy) > self.head_movement_threshold:
            return "nodding head"
            
        return ""

    def _detect_shoulder_movement(self, landmarks):
 
        left_shoulder = landmarks.landmark[self.mp_pose.PoseLandmark.LEFT_SHOULDER]
        right_shoulder = landmarks.landmark[self.mp_pose.PoseLandmark.RIGHT_SHOULDER]
        left_elbow = landmarks.landmark[self.mp_pose.PoseLandmark.LEFT_ELBOW]
        right_elbow = landmarks.landmark[self.mp_pose.PoseLandmark.RIGHT_ELBOW]
        
        # Store current shoulder positions
        shoulder_pos = (
            (left_shoulder.x, left_shoulder.y),
            (right_shoulder.x, right_shoulder.y)
        )
        self.shoulder_positions.append(shoulder_pos)
        
        if len(self.shoulder_positions) < 2:
            return ""
            
        # Calculate movement
        prev_pos = self.shoulder_positions[-2]
        left_dx = shoulder_pos[0][0] - prev_pos[0][0]
        left_dy = shoulder_pos[0][1] - prev_pos[0][1]
        right_dx = shoulder_pos[1][0] - prev_pos[1][0]
        right_dy = shoulder_pos[1][1] - prev_pos[1][1]
        
        # Calculate elbow positions for more context
        left_elbow_pos = (left_elbow.x, left_elbow.y)
        right_elbow_pos = (right_elbow.x, right_elbow.y)
        
        # Enhanced shoulder movement detection
        # Check vertical and horizontal movements with context
        vertical_diff_left = abs(left_dy)
        vertical_diff_right = abs(right_dy)
        horizontal_diff_left = abs(left_dx)
        horizontal_diff_right = abs(right_dx)
        
        # Shoulder shrug detection
        if (vertical_diff_left > self.shoulder_movement_threshold and 
            vertical_diff_right > self.shoulder_movement_threshold and 
            abs(left_dy - right_dy) < 0.02):  # Almost simultaneous
            return "shrugging shoulders"
        
        # Asymmetric shoulder movement
        if vertical_diff_left > self.shoulder_movement_threshold * 1.5 and \
           vertical_diff_right < self.shoulder_movement_threshold:
            return "raising left shoulder"
        
        if vertical_diff_right > self.shoulder_movement_threshold * 1.5 and \
           vertical_diff_left < self.shoulder_movement_threshold:
            return "raising right shoulder"
        
        # Rotational movement
        if horizontal_diff_left > self.shoulder_movement_threshold and \
           horizontal_diff_right > self.shoulder_movement_threshold:
            return "rotating shoulders"
        
        return ""

    def detect_posture(self, landmarks):
        """Enhanced posture detection with improved sitting and squatting logic"""
        current_time = time.time()
        
        # Only update posture every 5 seconds
        if (current_time - self.last_posture_change_time < self.posture_change_interval 
            and self.current_posture is not None):
            return self.current_posture

        # Get base posture
        base_posture = self._detect_base_posture(landmarks)
        
        if base_posture in "squatting" :
            
            send_number_to_nao(5)
        
        
        # Detect head and shoulder movements
        head_movement = self._detect_head_movement(landmarks)
        shoulder_movement = self._detect_shoulder_movement(landmarks)
        
        if head_movement in "nodding head" and base_posture in "standing":
            send_number_to_nao(5)
            
        if shoulder_movement in "shrugging shoulders" and base_posture in "standing":
            send_number_to_nao(7)
            
        if shoulder_movement in "raising left shoulder" and base_posture in "standing":
            send_number_to_nao(9) 
            
        if shoulder_movement in "raising right shoulder" and base_posture in "standing":
            send_number_to_nao(9)
        
        # Determine sitting type if sitting
        if 'sitting' in base_posture:
            base_posture = f"sitting on floor"
           
            send_number_to_nao(4)
        
        # Combine movements with base posture
        movements = []
        if head_movement:
            movements.append(head_movement)
          
        if shoulder_movement:
            movements.append(shoulder_movement)
          
        if movements:
            final_posture = f"{base_posture} while {' and '.join(movements)}"
        else:
            final_posture = base_posture
        
        
        # Update tracking
        self.current_posture = final_posture
        self.last_posture_change_time = current_time
        
        
        return final_posture

    def _detect_base_posture(self, landmarks):
        """Detect base posture (standing, sitting, squatting)"""
        # Get relevant landmark points
        left_shoulder = landmarks.landmark[self.mp_pose.PoseLandmark.LEFT_SHOULDER]
        right_shoulder = landmarks.landmark[self.mp_pose.PoseLandmark.RIGHT_SHOULDER]
        left_hip = landmarks.landmark[self.mp_pose.PoseLandmark.LEFT_HIP]
        right_hip = landmarks.landmark[self.mp_pose.PoseLandmark.RIGHT_HIP]
        left_knee = landmarks.landmark[self.mp_pose.PoseLandmark.LEFT_KNEE]
        right_knee = landmarks.landmark[self.mp_pose.PoseLandmark.RIGHT_KNEE]
        left_ankle = landmarks.landmark[self.mp_pose.PoseLandmark.LEFT_ANKLE]
        right_ankle = landmarks.landmark[self.mp_pose.PoseLandmark.RIGHT_ANKLE]
        
        # Calculate vertical positions and angles
        hip_y = (left_hip.y + right_hip.y) / 2
        knee_y = (left_knee.y + right_knee.y) / 2
        ankle_y = (left_ankle.y + right_ankle.y) / 2
        shoulder_y = (left_shoulder.y + right_shoulder.y) / 2
        
        # Calculate joint angles
        left_knee_angle = self._calculate_angle(
            (left_hip.x, left_hip.y),
            (left_knee.x, left_knee.y),
            (left_ankle.x, left_ankle.y)
        )
        right_knee_angle = self._calculate_angle(
            (right_hip.x, right_hip.y),
            (right_knee.x, right_knee.y),
            (right_ankle.x, right_ankle.y)
        )
        
        # Calculate hip angle
        hip_angle = self._calculate_angle(
            (shoulder_y, 0),
            (hip_y, 0),
            (knee_y, 0)
        )
        
        # Enhanced posture detection with confidence scores
        standing_score = self._calculate_standing_score(
            hip_y, knee_y, ankle_y, shoulder_y,
            left_knee_angle, right_knee_angle
        )
        sitting_score = self._calculate_sitting_score(
            hip_y, knee_y, ankle_y,
            left_knee_angle, right_knee_angle
        )
        squatting_score = self.calculate_squat_score(
            hip_y, knee_y, ankle_y, shoulder_y,
            left_knee_angle, right_knee_angle
        )

        # Adjust threshold for posture determination
        scores = {
            "standing": standing_score,
            "sitting": sitting_score,
            "squatting": squatting_score
        }

        # More nuanced posture selection
        base_posture = max(scores.items(), key=lambda x: x[1])[0]


        # Additional checks to prevent false positives
        if base_posture == "standing":
            # Check if it might actually be a squat
            if squatting_score > 0.00000001 or scores["squatting"] > scores["standing"]:
                base_posture = "squatting"
        
        if base_posture == "sitting":
            # Check if it might actually be a squat
            if squatting_score > 0.00000001 or scores["squatting"] > scores["sitting"]:
                base_posture = "squatting"
        
            # Check for raised arms
        if self._are_arms_raised(landmarks):
            base_posture += " with raised arms"

        return base_posture

    def _calculate_standing_score(self, hip_y, knee_y, ankle_y, shoulder_y,
                                 left_knee_angle, right_knee_angle):

        score = 0.0

        # 1. Vertical Alignment Check (40% of score)
        if shoulder_y > hip_y > knee_y > ankle_y:
            score += 0.4

        # 2. Knee Angle Check (30% of score)
        knee_angle_score = 0.0
        if 160 <= left_knee_angle <= 180 and 160 <= right_knee_angle <= 180:
            # Both knees in fully extended position
            knee_angle_score = 0.3
        elif 160 <= left_knee_angle <= 180 or 160 <= right_knee_angle <= 180:
            # At least one knee in fully extended position
            knee_angle_score = 0.15
        score += knee_angle_score

        # 3. Spine Alignment Check (30% of score)
        spine_alignment_score = 0.0
        if abs(shoulder_y - hip_y) < 0.1:
            # Shoulder and hip vertically aligned
            spine_alignment_score = 0.3
        elif abs(shoulder_y - hip_y) < 0.2:
            # Moderate spine alignment
            spine_alignment_score = 0.15
        score += spine_alignment_score

        return score

    def _calculate_sitting_score(self, hip_y, knee_y, ankle_y,
                               left_knee_angle, right_knee_angle):
        """Calculate confidence score for sitting posture"""
        score = 0.0
        
        # 1. Hip-Knee Alignment Check (40% of score)
        alignment_score = 0.0

        # Instead of comparing to floor height, check relative positioning
        if abs(knee_y - hip_y) < 0.1:  # Knees close to hip level
            alignment_score = 0.4
        score += alignment_score

        # 2. Knee Angle Check (30% of score)
        knee_angle_score = 0.0
        if 90 <= left_knee_angle <= 120 and 90 <= right_knee_angle <= 120:
            # Both knees in proper flexion range for floor sitting
            knee_angle_score = 0.3
        elif 90 <= left_knee_angle <= 120 or 90 <= right_knee_angle <= 120:
            # At least one knee in proper flexion range
            knee_angle_score = 0.15
        score += knee_angle_score

        # 3. Ankle Position Check (30% of score)
        ankle_score = 0.0
        if abs(ankle_y - knee_y) < 0.1:  # Ankles close to knee level
            ankle_score = 0.3
        score += ankle_score

        return min(score, 1.0)  # Ensure score doesn't exceed 1.0

    def calculate_squat_score(self, hip_y, knee_y, ankle_y, shoulder_y,
                               left_knee_angle, right_knee_angle):
        # Basic score calculation focusing on key body alignments
        score = 0.0

        # 1. Vertical Alignment Check (40% of score)
        if shoulder_y > hip_y > knee_y > ankle_y:
            score += 0.4

        # 2. Squat Depth Check (30% of score)
        depth = hip_y - ankle_y
        if depth > 0.5:  # Deep squat
            score += 0.3
        elif depth > 0.3:  # Moderate squat
            score += 0.2
        elif depth > 0.2:  # Partial squat
            score += 0.1

        # 3. Knee Angle Check (30% of score)
        knee_angle_score = 0.0
        if 50 < left_knee_angle < 110 and 50 < right_knee_angle < 110:
            knee_angle_score = 0.3
        elif 50 < left_knee_angle < 110 or 50 < right_knee_angle < 110:
            knee_angle_score = 0.15
        score += knee_angle_score

        return score
    
    def _are_arms_raised(self, landmarks):
        
        left_wrist = landmarks.landmark[self.mp_pose.PoseLandmark.LEFT_WRIST]
        right_wrist = landmarks.landmark[self.mp_pose.PoseLandmark.RIGHT_WRIST]
        left_shoulder = landmarks.landmark[self.mp_pose.PoseLandmark.LEFT_SHOULDER]
        right_shoulder = landmarks.landmark[self.mp_pose.PoseLandmark.RIGHT_SHOULDER]
        left_elbow = landmarks.landmark[self.mp_pose.PoseLandmark.LEFT_ELBOW]
        right_elbow = landmarks.landmark[self.mp_pose.PoseLandmark.RIGHT_ELBOW]

        # Check both elbow and wrist positions relative to shoulders
        left_arm_raised = (left_wrist.y < left_shoulder.y - 0.1 or 
                          left_elbow.y < left_shoulder.y - 0.1)
        right_arm_raised = (right_wrist.y < right_shoulder.y - 0.1 or 
                           right_elbow.y < right_shoulder.y - 0.1)

        return left_arm_raised or right_arm_raised

    def _calculate_hip_knee_ratio(self, left_hip, right_hip, left_knee, right_knee):
        """Calculate the ratio between hip and knee positions"""
        hip_y = (left_hip.y + right_hip.y) / 2
        knee_y = (left_knee.y + right_knee.y) / 2
        return abs(knee_y - hip_y)

    def _calculate_angle(self, p1, p2, p3):
        """Calculate angle between three points"""
        v1 = np.array([p1[0] - p2[0], p1[1] - p2[1]])
        v2 = np.array([p3[0] - p2[0], p3[1] - p2[1]])

        cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
        angle = np.arccos(np.clip(cos_angle, -1.0, 1.0))
        return np.degrees(angle)



    def calculate_bounding_box(self, landmarks, model_type):

        if model_type == 'mediapipe':
            points = [(lm.x, lm.y) for lm in landmarks.landmark]
        else:  # YOLO
            points = [(x/self.current_frame_shape[1], y/self.current_frame_shape[0]) 
                     for x, y, _ in landmarks]

        x_coords = [p[0] for p in points]
        y_coords = [p[1] for p in points]

        # Calculate normalized coordinates
        x_min, x_max = min(x_coords), max(x_coords)
        y_min, y_max = min(y_coords), max(y_coords)

        # Convert to pixel coordinates
        return {
            'x1': int(x_min * self.current_frame_shape[1]),
            'y1': int(y_min * self.current_frame_shape[0]),
            'x2': int(x_max * self.current_frame_shape[1]),
            'y2': int(y_max * self.current_frame_shape[0])
        }

    def draw_action_box(self, frame, bbox, action):
        """Draw bounding box and action label on frame"""
        # Draw bounding box
        cv2.rectangle(frame, 
                     (bbox['x1'], bbox['y1']), 
                     (bbox['x2'], bbox['y2']), 
                     (0, 255, 0), 2)

        # Draw action label
        label_size = cv2.getTextSize(action, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)[0]
        cv2.rectangle(frame,
                     (bbox['x1'], bbox['y1'] - label_size[1] - 10),
                     (bbox['x1'] + label_size[0], bbox['y1']),
                     (0, 255, 0), -1)
        cv2.putText(frame, action,
                   (bbox['x1'], bbox['y1'] - 5),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)
    
    def _check_confidence(self, confidence):

        if confidence > self.low_confidence_threshold:
            # Reset low confidence counter
            self.low_confidence_count = 0
            return "normal"
        
        # Increment low confidence counter
        self.low_confidence_count += 1
        
        if confidence <= self.very_low_confidence_threshold:
            # Critical low confidence
            send_number_to_nao(1)
            return "very_low"
        
        if self.low_confidence_count >= self.max_low_confidence_attempts:
            send_number_to_nao(1)  
            self.low_confidence_count = 0
            return "low"
    
 

            

In [27]:
import tkinter as tk
from tkinter import ttk

def create_instruction_window():
    # Create the main window
    window = tk.Tk()
    window.title("NAO Robot Controls")
    window.geometry("300x250")
    
    # Create main frame with padding
    main_frame = ttk.Frame(window, padding="20")
    main_frame.pack(fill="both", expand=True)
    
    # Title
    title_label = ttk.Label(
        main_frame,
        text="Keyboard Controls",
        font=('Helvetica', 14, 'bold')
    )
    title_label.pack(pady=(0, 20))
    
    # Instructions frame
    instructions_frame = ttk.Frame(main_frame)
    instructions_frame.pack(fill="both", expand=True)
    
    # Keyboard shortcuts
    shortcuts = [
        ("Q or ESC", "Quit application"),
        ("R", "Start/Stop recording"),
        ("S", "Save current frame")
    ]
    
    # Create shortcuts display
    for key, description in shortcuts:
        # Key container
        key_frame = ttk.Frame(instructions_frame)
        key_frame.pack(fill="x", pady=10)
        
        # Key binding with custom style
        key_label = ttk.Label(
            key_frame,
            text=key,
            background='lightgray',
            relief='raised',
            padding=5,
            width=10
        )
        key_label.pack(side="left", padx=(0, 10))
        
        # Description
        desc_label = ttk.Label(
            key_frame,
            text=description,
            wraplength=200
        )
        desc_label.pack(side="left", fill="x", expand=True)
    
    return window

if __name__ == "__main__":
    window = create_instruction_window()
    window.mainloop()

In [28]:
SERVER_IP = "127.0.0.1"
SERVER_PORT = 9999
FRAME_BUFFER_SIZE = 5  # Number of frames to buffer

class NAOPoseDetectionApp:
    def __init__(self, root):
        self.root = root
        self.root.title("NAO Robot Pose Detection")
        
        # Make it fullscreen
        self.root.attributes('-fullscreen', True)
        
        # Configure root to expand
        self.root.grid_rowconfigure(0, weight=1)
        self.root.grid_columnconfigure(1, weight=3)  # Main video feed gets more space
        
        # Create main frame for video
        self.main_frame = ttk.Frame(root)
        self.main_frame.grid(row=0, column=1, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        # Create analytics panel
        self.analytics_frame = ttk.Frame(root)
        self.analytics_frame.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S), padx=5)
        
        # Configure frames to expand
        self.main_frame.grid_rowconfigure(0, weight=1)
        self.main_frame.grid_columnconfigure(0, weight=1)
        
        # Create label for image display
        self.image_label = ttk.Label(self.main_frame)
        self.image_label.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        # Create status and control panel
        self.control_frame = ttk.Frame(self.main_frame)
        self.control_frame.grid(row=1, column=0, pady=5)
        
        # Status indicators
        self.status_label = ttk.Label(self.control_frame, text="Disconnected")
        self.status_label.pack(side=tk.LEFT, padx=5)
        
        self.fps_label = ttk.Label(self.control_frame, text="FPS: 0")
        self.fps_label.pack(side=tk.LEFT, padx=5)
        
        # Recording indicator
        self.recording_label = ttk.Label(self.control_frame, text="⚫", foreground="gray")
        self.recording_label.pack(side=tk.LEFT, padx=5)
        
        # Setup analytics panel
        self.setup_analytics_panel()
        
        # Frame processing queue and buffer
        self.frame_queue = queue.Queue(maxsize=FRAME_BUFFER_SIZE)
        self.last_frames_time = deque(maxlen=30)
        
        # Analytics data storage
        self.posture_history = deque(maxlen=100)
        self.confidence_history = deque(maxlen=100)
        self.fps_history = deque(maxlen=100)
        
        # Connection status
        self.running = False
        self.client_socket = None
        self._quit_event = threading.Event()
        
        self.pose_detector = PoseDetector()
        
        # Bind keys
        self.setup_key_bindings()
        
        # Socket configuration
        self.socket_timeout = 5.0
        self.tcp_nodelay = True
        
        # Start threads
        self.start_threads()
        
        # Schedule first analytics update
        self.root.after(1000, self.update_analytics)

    def setup_analytics_panel(self):
        
        # Current Posture
        self.posture_frame = ttk.LabelFrame(self.analytics_frame, text="Current Posture")
        self.posture_frame.pack(fill=tk.X, padx=5, pady=5)
        self.current_posture_label = ttk.Label(self.posture_frame, text="No pose detected")
        self.current_posture_label.pack(padx=5, pady=5)
        
        # Confidence Score
        self.confidence_frame = ttk.LabelFrame(self.analytics_frame, text="Confidence Score")
        self.confidence_frame.pack(fill=tk.X, padx=5, pady=5)
        self.confidence_progressbar = ttk.Progressbar(self.confidence_frame, length=200, mode='determinate')
        self.confidence_progressbar.pack(padx=5, pady=5)
        
        # Performance Metrics
        self.metrics_frame = ttk.LabelFrame(self.analytics_frame, text="Performance Metrics")
        self.metrics_frame.pack(fill=tk.X, padx=5, pady=5)
        self.avg_fps_label = ttk.Label(self.metrics_frame, text="Avg FPS: 0")
        self.avg_fps_label.pack(padx=5, pady=2)
        self.process_time_label = ttk.Label(self.metrics_frame, text="Process Time: 0 ms")
        self.process_time_label.pack(padx=5, pady=2)
        
        # Create matplotlib figure for real-time plots
        self.fig = Figure(figsize=(4, 3), dpi=100)
        self.ax = self.fig.add_subplot(111)
        self.canvas = FigureCanvasTkAgg(self.fig, master=self.analytics_frame)
        self.canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True, padx=5, pady=5)

    def setup_key_bindings(self):
        
        self.root.bind('q', self.quit_application)
        self.root.bind('Q', self.quit_application)
        self.root.bind('<Escape>', self.quit_application)
        self.root.bind('r', self.toggle_recording)
        self.root.bind('s', self.save_frame)
        
    def start_threads(self):
        
        self.connection_thread = threading.Thread(target=self.receive_frames)
        self.processing_thread = threading.Thread(target=self.process_frames)
        self.connection_thread.daemon = True
        self.processing_thread.daemon = True
        self.connection_thread.start()
        self.processing_thread.start()

    def update_analytics(self):
       
        if not self._quit_event.is_set():
            try:
                # Update confidence progressbar
                if self.confidence_history:
                    confidence = self.confidence_history[-1]
                    self.confidence_progressbar['value'] = confidence * 100
                
                # Update performance metrics
                if self.fps_history:
                    avg_fps = np.mean(list(self.fps_history))
                    self.avg_fps_label.configure(text=f"Avg FPS: {avg_fps:.1f}")
                
                # Update real-time plot
                self.ax.clear()
                if self.confidence_history:
                    self.ax.plot(list(self.confidence_history), label='Confidence')
                    self.ax.set_ylim(0, 1)
                    self.ax.set_title('Confidence Over Time')
                    self.ax.legend()
                    self.canvas.draw()
                
                # Schedule next update
                self.root.after(1000, self.update_analytics)
                
            except Exception as e:
                logging.error(f"Error updating analytics: {e}")

    def toggle_recording(self, event=None):
        
        self.pose_detector.toggle_recording()
        self.recording_label.configure(
            text="🔴",
            foreground="red" if self.pose_detector.recording else "gray"
        )

    def save_frame(self, event=None):
        
        self.pose_detector.save_frame()

    def update_fps(self):
      
        if len(self.last_frames_time) >= 2:
            fps = len(self.last_frames_time) / (self.last_frames_time[-1] - self.last_frames_time[0])
            self.fps_label.configure(text=f"FPS: {fps:.1f}")
            self.fps_history.append(fps)
        self.root.after(1000, self.update_fps)

    def process_frames(self):
        
        while not self._quit_event.is_set():
            try:
                frame = self.frame_queue.get(timeout=0.1)
                start_time = time.time()
                
                # Process frame with pose detection
                processed_frame = self.pose_detector.process_frame(frame)
                
                # Update analytics
                if hasattr(self.pose_detector, 'current_posture'):
                    self.posture_history.append(self.pose_detector.current_posture)
                    self.root.after(0, self.current_posture_label.configure, 
                                  {'text': self.pose_detector.current_posture})
                
                if hasattr(self.pose_detector, 'current_confidence'):
                    self.confidence_history.append(self.pose_detector.current_confidence)
                
                # Update processing time
                process_time = (time.time() - start_time) * 1000
                self.root.after(0, self.process_time_label.configure, 
                              {'text': f"Process Time: {process_time:.1f} ms"})
                
                # Convert and display frame
                self.display_frame(processed_frame)
                
                self.last_frames_time.append(time.time())
                
            except queue.Empty:
                continue
            except Exception as e:
                if not self._quit_event.is_set():
                    logging.error(f"Error processing frame: {e}")

    def display_frame(self, frame):
        
        try:
            # Convert to PIL Image
            pil_image = PILImage.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            
            # Get screen dimensions
            screen_width = self.root.winfo_width()
            screen_height = self.root.winfo_height()
            
            # Resize image efficiently
            img_width, img_height = pil_image.size
            aspect_ratio = img_width / img_height
            
            if screen_width / screen_height > aspect_ratio:
                new_height = screen_height
                new_width = int(screen_height * aspect_ratio)
            else:
                new_width = screen_width
                new_height = int(screen_width / aspect_ratio)
            
            pil_image = pil_image.resize((new_width, new_height), PILImage.Resampling.NEAREST)
            
            # Convert to PhotoImage
            photo = ImageTk.PhotoImage(image=pil_image)
            
            # Update label
            self.root.after(0, self.update_image_label, photo)
            
        except Exception as e:
            logging.error(f"Error displaying frame: {e}")

    def update_image_label(self, photo):
        
        if not self._quit_event.is_set():
            self.image_label.configure(image=photo)
            self.image_label.image = photo

    def receive_frames(self):
        
        retry_count = 3
        retry_delay = 2  # seconds between retries

        while retry_count > 0 and not self._quit_event.is_set():
            self.client_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)


            self.client_socket.settimeout(self.socket_timeout)


            if self.tcp_nodelay:
                self.client_socket.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)


            self.running = True

            try:
                # Try to connect to both sockets
                self.status_label.configure(text="Attempting to connect to server...")
                self.client_socket.connect((SERVER_IP, SERVER_PORT))


                self.status_label.configure(text="Connected to NAO robot stream server")
                self.update_fps()
                retry_count = 3 

                while self.running and not self._quit_event.is_set():
                    try:
                        frame = self.receive_single_frame()
                        if frame is not None:
                            try:
                                self.frame_queue.put_nowait(frame)
                            except queue.Full:
                                continue
                    except socket.timeout:
                        continue
                    except Exception as e:
                        if not self._quit_event.is_set():
                            logging.error(f"Error receiving frame: {e}")
                            self.status_label.configure(text=f"Connection error: {str(e)}")
                        break

            except ConnectionRefusedError:
                retry_count -= 1
                if retry_count > 0:
                    self.status_label.configure(text=f"Connection refused. Retrying in {retry_delay} seconds... ({retry_count} attempts left)")
                    time.sleep(retry_delay)
                    continue
                else:
                    self.status_label.configure(text="Could not connect to server. Make sure the server is running.")
            except Exception as e:
                if not self._quit_event.is_set():
                    logging.error(f"Unexpected error: {e}")
                    self.status_label.configure(text=f"Unexpected error: {str(e)}")
            finally:
                self.cleanup()

            if retry_count == 0:
                break

    def receive_single_frame(self):
       
        # Receive frame dimensions
        width_data = self.client_socket.recv(4)
        if not width_data:
            raise ConnectionError("Server disconnected")
        
        width = int.from_bytes(width_data, byteorder="big")
        height = int.from_bytes(self.client_socket.recv(4), byteorder="big")
        
        # Receive frame data efficiently
        frame_size = width * height * 3
        raw_data = bytearray(frame_size)
        view = memoryview(raw_data)
        remaining = frame_size
        
        while remaining > 0 and not self._quit_event.is_set():
            received = self.client_socket.recv_into(view, remaining)
            if not received:
                raise ConnectionError("Connection lost while receiving frame data")
            view = view[received:]
            remaining -= received
        
        if self._quit_event.is_set():
            return None
        
        # Convert to numpy array efficiently
        return np.frombuffer(raw_data, dtype=np.uint8).reshape((height, width, 3))

    def cleanup(self):
       
        if self.client_socket:
            try:
                self.client_socket.close()
            except:
                pass
        if not self._quit_event.is_set():
            self.status_label.configure(text="Connection closed")

    def quit_application(self, event=None):
      
        if self._quit_event.is_set():
            return
            
        try:
            # Generate final report before shutting down
            self.generate_final_report()
        except Exception as e:
            logging.error(f"Error generating final report during shutdown: {e}")
            
        self._quit_event.set()
        self.running = False
        
        def force_quit():
            self.cleanup()
            self.root.quit()
            self.root.destroy()
        
        self.root.after(100, force_quit)
        
        
    def generate_final_report(self):

        report_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        report_dir = Path(f'reports/session_{report_timestamp}')
        report_dir.mkdir(parents=True, exist_ok=True)

        # Generate analytics report using PoseAnalytics
        self.pose_detector.analytics.generate_analytics_report()

        # Create session summary plots
        self._generate_session_plots(report_dir)

        # Generate comprehensive text report
        self._generate_text_report(report_dir, report_timestamp)

    def _generate_session_plots(self, report_dir):
       
        # Posture Distribution Plot
        plt.figure(figsize=(12, 6))
        posture_counts = pd.Series(list(self.posture_history)).value_counts()
        sns.barplot(x=posture_counts.values, y=posture_counts.index)
        plt.title('Posture Distribution Throughout Session')
        plt.xlabel('Count')
        plt.tight_layout()
        plt.savefig(report_dir / 'session_posture_distribution.png')
        plt.close()

        # Performance Metrics Plot
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

        # FPS over time
        fps_times = list(range(len(self.fps_history)))
        ax1.plot(fps_times, list(self.fps_history), label='FPS')
        ax1.set_title('FPS Over Time')
        ax1.set_ylabel('Frames Per Second')
        ax1.grid(True)

        # Confidence over time
        confidence_history = list(self.confidence_history)
        confidence_times = list(range(len(confidence_history)))

        # Ensure alignment by truncating the longer dataset
        if len(confidence_times) > len(fps_times):
            confidence_times = confidence_times[:len(fps_times)]
            confidence_history = confidence_history[:len(fps_times)]
        elif len(fps_times) > len(confidence_times):
            fps_times = fps_times[:len(confidence_times)]

        # Log any remaining mismatch (unlikely after alignment)
        if len(confidence_times) != len(confidence_history):
            logging.warning(f"Final dimension mismatch: confidence_times({len(confidence_times)}) vs. confidence_history({len(confidence_history)})")

        ax2.plot(confidence_times, confidence_history, label='Confidence', color='green')
        ax2.set_title('Detection Confidence Over Time')
        ax2.set_ylabel('Confidence Score')
        ax2.set_xlabel('Frame Number')
        ax2.grid(True)

        plt.tight_layout()
        plt.savefig(report_dir / 'session_performance_metrics.png')
        plt.close()


    def _generate_text_report(self, report_dir, timestamp):
        
        report_path = report_dir / f'session_report_{timestamp}.txt'
        
        with open(report_path, 'w') as f:
            # Header
            f.write("=== NAO Robot Pose Detection Session Report ===\n")
            f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")

            # Session Overview
            session_duration = (datetime.now() - self.pose_detector.analytics.current_metrics['session_start'])
            f.write("Session Overview:\n")
            f.write(f"Duration: {str(session_duration)}\n")
            f.write(f"Total Frames Processed: {len(self.fps_history)}\n")
            f.write(f"Average FPS: {np.mean(list(self.fps_history)):.2f}\n\n")

            # Pose Detection Statistics
            f.write("Pose Detection Statistics:\n")
            if self.posture_history:
                posture_counts = pd.Series(list(self.posture_history), dtype="str").value_counts()
            else:
                posture_counts = pd.Series([], dtype="str")
            f.write("Posture Distribution:\n")
            for posture, count in posture_counts.items():
                f.write(f"- {posture}: {count} frames ({count/len(self.posture_history)*100:.1f}%)\n")
            f.write("\n")

            # Performance Metrics
            f.write("Performance Metrics:\n")
            if self.confidence_history:
                conf_array = np.array(list(self.confidence_history))
                f.write(f"Average Confidence Score: {np.mean(conf_array):.3f}\n")
                f.write(f"Minimum Confidence Score: {np.min(conf_array):.3f}\n")
                f.write(f"Maximum Confidence Score: {np.max(conf_array):.3f}\n")
                f.write(f"Confidence Score Std Dev: {np.std(conf_array):.3f}\n\n")
            else:
                f.write("No confidence scores available.\n\n")

            # Detection Quality Analysis
            low_conf_frames = np.sum(conf_array < 0.5)
            f.write("Detection Quality Analysis:\n")
            f.write(f"Low Confidence Detections (<50%): {low_conf_frames} frames ")
            f.write(f"({low_conf_frames/len(conf_array)*100:.1f}% of total)\n")
            
            # File Information
            f.write("\nGenerated Files:\n")
            f.write("1. Analytics Plots:\n")
            f.write("   - session_posture_distribution.png\n")
            f.write("   - session_performance_metrics.png\n")
            f.write("2. Raw Analytics Data: analytics_report_[timestamp].txt\n")
            f.write("3. Session Log: nao_pose_detection_[timestamp].log\n\n")

            # Report Footer
            f.write("=== End of Report ===\n")

def main():

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(f'nao_pose_detection_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'),
            logging.StreamHandler()
        ]
    )
    
    try:
        root = tk.Tk()
        app = NAOPoseDetectionApp(root)
        root.mainloop()
    except Exception as e:
        logging.error(f"Application error: {e}")
        try:
            # Try to generate final report even if there's an error
            app.generate_final_report()
        except:
            logging.error("Could not generate final report after error")
        sys.exit(1)
    finally:
        logging.info("Application shutdown")

if __name__ == "__main__":
    main()

C:\Users\kitmi\anaconda3\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
2025-01-02 17:30:55,339 - WARNING - Low confidence detection: 0.44
2025-01-02 17:30:55,392 - WARNING - Low confidence detection: 0.44
2025-01-02 17:30:55,456 - WARNING - Low confidence detection: 0.44
2025-01-02 17:30:55,510 - WARNING - Low confidence detection: 0.44
2025-01-02 17:30:55,559 - WARNING - Low confidence detection: 0.44
2025-01-02 17:30:57,659 - ERROR - Error in process_frame: [WinError 10061] No connection could be made because the target machine actively refused it
2025-01-02 17:31:00,028 - INFO - Application shutdown
2025-01-02 17:31:01,259 - ERROR - Error processing landmarks: [WinError 10061] No connection could be made because the target machine actively r

In [59]:
#send_number_to_nao(0)

Sent number: 4
